In [5]:
import tomllib
from pathlib import Path
import psycopg2

with open("../../../.streamlit/secrets.toml", "rb") as file:
    secrets = tomllib.load(file)

database = secrets["database"]

connection = psycopg2.connect(
    dbname=database["database"],
    user=database["username"],
    password=database["password"],
    host=database["host"],
    port=database["port"],
    sslmode=database["sslmode"],
)

print("Connected to PostgreSQL")

Connected to PostgreSQL


In [5]:
create_table_query = """
CREATE TABLE ai_usage_records (
    id BIGSERIAL PRIMARY KEY,
    operation TEXT NOT NULL,
    model TEXT NOT NULL,
    prompt_tokens INTEGER NOT NULL,
    completion_tokens INTEGER NOT NULL,
    total_tokens INTEGER NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP,
    user_id TEXT,
    project_id TEXT
);
"""

with connection.cursor() as cursor:
    cursor.execute(create_table_query)

connection.commit()

print("Table created successfully")

Table created successfully


In [6]:
with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM ai_usage_records LIMIT 0")
    columns = [column.name for column in cursor.description]

print(columns)

connection.close()

['id', 'operation', 'model', 'prompt_tokens', 'completion_tokens', 'total_tokens', 'created_at', 'user_id', 'project_id']


In [3]:
migration_query = """
ALTER TABLE ai_usage_records
    ADD COLUMN input_tokens INTEGER,
    ADD COLUMN output_tokens INTEGER,
    ADD COLUMN input_text_tokens INTEGER,
    ADD COLUMN input_image_tokens INTEGER,
    ADD COLUMN output_text_tokens INTEGER,
    ADD COLUMN output_image_tokens INTEGER;

UPDATE ai_usage_records
SET
    input_tokens = prompt_tokens,
    output_tokens = completion_tokens;

ALTER TABLE ai_usage_records
    ALTER COLUMN input_tokens SET NOT NULL,
    ALTER COLUMN output_tokens SET NOT NULL;

ALTER TABLE ai_usage_records
    DROP COLUMN prompt_tokens,
    DROP COLUMN completion_tokens;
"""

with connection:
    with connection.cursor() as cursor:
        cursor.execute(migration_query)

print("AI usage table migrated successfully.")

AI usage table migrated successfully.


In [4]:
with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM ai_usage_records LIMIT 0")
    columns = [column.name for column in cursor.description]

print(columns)

['id', 'operation', 'model', 'total_tokens', 'created_at', 'user_id', 'project_id', 'input_tokens', 'output_tokens', 'input_text_tokens', 'input_image_tokens', 'output_text_tokens', 'output_image_tokens']


In [ ]:
with connection.cursor() as cursor:
    cursor.execute("""
        SELECT
            id,
            operation,
            model,
            input_tokens,
            output_tokens,
            total_tokens,
            project_id,
            created_at
        FROM ai_usage_records
        WHERE project_id = '2'
        ORDER BY created_at DESC;
    """)

    records = cursor.fetchall()

    for record in records:
        print(record)

connection.close()

UndefinedFunction: operator does not exist: text = integer
LINE 12:         WHERE project_id = 2
                                  ^
HINT:  No operator matches the given name and argument types. You might need to add explicit type casts.
